# Marine account analysis

## Purpose and scope

This notebook inspects the marine side of the national accounts and the selected update to the mangrove-area input. It focuses on the current GMW 2025 area assigned to study units, its coverage and reconciliation, and the resulting marine account summaries. It does not regenerate the spatial overlay; the overlay and reallocation scripts remain the reproducible source for that operation.

## What is included

The current update replaces the mangrove area input with the spatially allocated GMW 2025 area. Other marine components and their published source rates remain those used by the account pipeline. The notebook displays those boundaries so the area update is not mistaken for a complete recalculation of every marine component.

## 1. Set up and load current marine outputs

The canonical unit table contains the applied GMW mangrove area and its account value. The validation summary records the source total, corrected total, area assigned to study units, and residual area that could not be assigned.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Run this notebook from within the repository.')

DATA = ROOT / 'data' / 'derived'
RESULTS = ROOT / 'results'
TABLES = RESULTS / 'tables' / 'manuscript'
VALIDATION = RESULTS / 'validation'
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
print(f'Repository: {ROOT}')

Repository: C:\Users\aurel\Desktop\M2\Stage ecoseas-csm\CSM\BNC2 paper\repo


## 2. Inspect the GMW area reconciliation

The source summary is a global check. The country/territory table is the applied, unit-level result. Compare assigned plus unassigned area with the corrected source total, allowing for documented rounding and spatial allocation residuals.

In [2]:
accounts = pd.read_csv(TABLES / 'Supplementary_country_level_accounts_2024.csv')
gmw_summary = pd.read_csv(VALIDATION / 'gmw_2025_corrected_area_summary.csv')
mangrove_audit = pd.read_csv(TABLES / 'Supplementary_mangrove_reconciliation.csv')

area_col = 'gmw_2025_mangrove_area_applied_ha'
mangrove_value_col = 'bcw_gmw_mangroves_usd2015_yr_mean'
unit_area = pd.to_numeric(accounts[area_col], errors='coerce')
unit_value = pd.to_numeric(accounts[mangrove_value_col], errors='coerce')
print('Study units with an applied GMW area:', int(unit_area.notna().sum()))
print('Applied area summed across unit records (ha):', float(unit_area.sum()))
display(gmw_summary.T.rename(columns={0: 'value'}))
display(mangrove_audit.head(12))

Study units with an applied GMW area: 267
Applied area summed across unit records (ha): 14728336.512591964


,value
vector_raw_ha,1.506748e+07
published_corrected_total_ha,1.472836e+07
documented_factor,9.775000e-01
observed_country_factor_min,9.774933e-01
observed_country_factor_max,9.774933e-01
spatial_assigned_corrected_ha,1.472834e+07
spatial_unassigned_corrected_ha,2.774863e+01
conservation_residual_ha,8.463860e-06
spatial_accounting_units,1.270000e+02
source_country_codes,1.290000e+02


,UNIT_CODE,ISO3,name,region,iso3,fra_mangroves_2025_ha,documented_forest_fraction,documented_forest_mangrove_area_ha,documented_other_wooded_land_fraction,documented_other_wooded_land_area_ha,documented_remaining_land_fraction,documented_remaining_land_area_ha,unresolved_mangrove_area_ha,overlap_status,desk_study,fra_report_handle,fra_report_url,evidence_note,report_excerpt,gmw_2025_mangrove_area_ha,groups,fra_total_forest_area_2025_ha,fra_mangrove_category_2025_ha,fra_other_wooded_land_2025_ha,gmw_2025_mangrove_area_applied_ha,gmw_mangrove_uptake_tC_yr_mean,gmw_mangrove_uptake_tCO2_yr_mean,gmw_mangrove_uptake_tC_yr_p05,gmw_mangrove_uptake_tC_yr_p50,gmw_mangrove_uptake_tC_yr_p95,bcw_gmw_mangroves_usd2015_yr_mean,bcw_usd2015_yr_mean,accounting_note
0,ABW,ABW,Aruba,Caribbean,ABW,70.0,NaN,0.0,NaN,0.0,NaN,0.0,70.0,no_exact_forest_split_verified,Yes,A03278EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,Original data | The forest area is estimated u...,1.130099e+02,SIDS,490.0,70.0,490.0,1.130099e+02,2.093692e+02,7.676871e+02,26.377876,1.295200e+02,6.547082e+02,4.274381e+05,1.529487e+07,FRA mangrove hectares are removed from GCW onl...
1,AGO,AGO,Angola,Africa,AGO,27920.0,NaN,0.0,NaN,0.0,NaN,0.0,27920.0,no_exact_forest_split_verified,No,A03636FR,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,NaN,2.857551e+04,LDCs,63262330.0,27920.0,63206690.0,2.857551e+04,5.294080e+04,1.941163e+05,6669.871979,3.275025e+04,1.655486e+05,1.080814e+08,2.499569e+10,FRA mangrove hectares are removed from GCW onl...
2,AIA,AIA,Anguilla,Caribbean,AIA,90.0,NaN,0.0,NaN,0.0,NaN,0.0,90.0,no_exact_forest_split_verified,Yes,A03275EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,NaN,8.061440e+00,SIDS,5500.0,90.0,NaN,8.061440e+00,1.493513e+01,5.476215e+01,1.881638,9.239176e+00,4.670293e+01,3.049085e+04,6.775158e+06,FRA mangrove hectares are removed from GCW onl...
3,ARE,ARE,United Arab Emirates,Asia,ARE,13610.0,NaN,0.0,NaN,0.0,NaN,0.0,13610.0,no_exact_forest_split_verified,Yes,A03544EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,2003 | Date palm 185.30 0.000 % 0.000 % 100.00...,9.658275e+03,Developing economies,324580.0,13610.0,310970.0,9.658275e+03,1.789353e+04,6.560961e+04,2254.358999,1.106930e+04,5.595399e+04,3.653057e+07,5.120821e+08,FRA mangrove hectares are removed from GCW onl...
4,ASM,ASM,American Samoa,Oceania,ASM,20.0,NaN,0.0,NaN,0.0,NaN,0.0,20.0,no_exact_forest_split_verified,No,A03274EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,NaN,3.346779e+01,SIDS,15850.0,20.0,15850.0,3.346779e+01,6.200453e+01,2.273499e+02,7.811788,3.835726e+01,1.938914e+02,1.265855e+05,2.073103e+08,FRA mangrove hectares are removed from GCW onl...
5,ATF-4,NaN,Europa Island,Africa,ATF-4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.644516e+02,Developing economies,NaN,NaN,NaN,6.644516e+02,1.231005e+03,4.513685e+03,155.091086,7.615247e+02,3.849416e+03,2.513160e+06,2.513160e+06,FRA mangrove hectares are removed from GCW onl...
6,ATF_GI,NaN,ATF_GI,NaN,ATF_GI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,NaN,FRA mangrove hectares are removed from GCW onl...
7,ATG,ATG,Antigua and Barbuda,Caribbean,ATG,840.0,NaN,0.0,NaN,0.0,NaN,0.0,840.0,no_exact_forest_split_verified,No,A03276EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,"Cactus scrub Open forest (10-40% canopy cover,...",9.526774e+02,SIDS,7790.0,840.0,7790.0,9.526774e+02,1.764990e+03,6.471631e+03,222.366493,1.091859e+03,5.519215e+03,3.603319e+06,5.605168e+08,FRA mangrove hectares are removed from GCW onl...
8,AUS,AUS,Australia,Oceania,AUS,1110000.0,NaN,0.0,NaN,0.0,NaN,0.0,1110000.

## 3. Review the unit-level distribution

The table below ranks study units by applied GMW area and reports the associated mangrove component value. This is a data inspection step: the ranking helps find high-area records and crosswalk issues, while interpretation of national responsibility remains a separate author decision.

In [3]:
unit_view = accounts.loc[unit_area.gt(0), ['UNIT_CODE', 'name', area_col, mangrove_value_col]].copy()
unit_view[area_col] = pd.to_numeric(unit_view[area_col], errors='coerce')
unit_view[mangrove_value_col] = pd.to_numeric(unit_view[mangrove_value_col], errors='coerce')
display(unit_view.sort_values(area_col, ascending=False).head(15))

,UNIT_CODE,name,gmw_2025_mangrove_area_applied_ha,bcw_gmw_mangroves_usd2015_yr_mean
113,IDN,Indonesia,3.146650e+06,1.190160e+10
40,BRA,Brazil,1.076951e+06,4.073361e+09
20,AUS,Australia,1.069096e+06,4.043649e+09
174,NGA,Nigeria,7.925423e+05,2.997639e+09
153,MEX,Mexico,7.901547e+05,2.988608e+09
168,MYS,Malaysia,5.596047e+05,2.116597e+09
158,MMR,Myanmar,5.583609e+05,2.111893e+09
189,PNG,Papua New Guinea,4.877654e+05,1.844879e+09
115,IND,India,4.671322e+05,1.766838e+09
30,BGD,Bangladesh,4.280579e+05,1.619047e+09


## 4. Compare the global account totals

Table 1 reports the global GCW, BCW and NCW estimates under the current accounting version. This cell displays the marine component alongside the forest and combined account so readers can trace which published table carries the totals.

In [4]:
global_table = pd.read_csv(TABLES / 'Table1_global_carbon_wealth.csv')
display(global_table)

,component,value_mean_trillion_usd2015_yr,value_p05_trillion_usd2015_yr,value_p50_trillion_usd2015_yr,value_p95_trillion_usd2015_yr,share_of_ncw_pct,gcw_records,bcw_records_including_source_aggregate,bcw_country_or_territory_records,units_with_both_accounts,records_with_either_account,aggregation_scope
0,Green carbon wealth,8.261443,3.226506,7.481159,14.999199,78.008686,236,180,179,174,242,Available component total across all GCW units.
1,Blue carbon wealth,2.328971,0.908838,2.113162,4.297896,21.991314,236,180,179,174,242,Available component total; BCW includes the so...
2,Natural carbon wealth,10.590414,4.163492,9.600348,19.204331,100.000000,236,180,179,174,242,Global GCW + global BCW; national NCW remains ...


## 5. Cross-source and boundary checks

Use the reconciliation table to inspect units with source-code mismatches or unassigned area. These are coverage and boundary diagnostics, not proof that a location has no mangroves. Keep the source map, allocation boundary, and FRA national report categories distinct when reviewing a discrepancy.

In [5]:
audit_cols = [c for c in mangrove_audit.columns if any(k in c.lower() for k in ['iso', 'name', 'area', 'status', 'fra', 'gmw', 'overlap', 'note'])]
display(mangrove_audit[audit_cols].head(30))

,ISO3,name,iso3,fra_mangroves_2025_ha,documented_forest_fraction,documented_forest_mangrove_area_ha,documented_other_wooded_land_fraction,documented_other_wooded_land_area_ha,documented_remaining_land_fraction,documented_remaining_land_area_ha,unresolved_mangrove_area_ha,overlap_status,fra_report_handle,fra_report_url,evidence_note,gmw_2025_mangrove_area_ha,fra_total_forest_area_2025_ha,fra_mangrove_category_2025_ha,fra_other_wooded_land_2025_ha,gmw_2025_mangrove_area_applied_ha,gmw_mangrove_uptake_tC_yr_mean,gmw_mangrove_uptake_tCO2_yr_mean,gmw_mangrove_uptake_tC_yr_p05,gmw_mangrove_uptake_tC_yr_p50,gmw_mangrove_uptake_tC_yr_p95,bcw_gmw_mangroves_usd2015_yr_mean,accounting_note
0,ABW,Aruba,ABW,70.0,NaN,0.0,NaN,0.0,NaN,0.0,70.0,no_exact_forest_split_verified,A03278EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,1.130099e+02,490.0,70.0,490.0,1.130099e+02,2.093692e+02,7.676871e+02,26.377876,1.295200e+02,6.547082e+02,4.274381e+05,FRA mangrove hectares are removed from GCW onl...
1,AGO,Angola,AGO,27920.0,NaN,0.0,NaN,0.0,NaN,0.0,27920.0,no_exact_forest_split_verified,A03636FR,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,2.857551e+04,63262330.0,27920.0,63206690.0,2.857551e+04,5.294080e+04,1.941163e+05,6669.871979,3.275025e+04,1.655486e+05,1.080814e+08,FRA mangrove hectares are removed from GCW onl...
2,AIA,Anguilla,AIA,90.0,NaN,0.0,NaN,0.0,NaN,0.0,90.0,no_exact_forest_split_verified,A03275EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,8.061440e+00,5500.0,90.0,NaN,8.061440e+00,1.493513e+01,5.476215e+01,1.881638,9.239176e+00,4.670293e+01,3.049085e+04,FRA mangrove hectares are removed from GCW onl...
3,ARE,United Arab Emirates,ARE,13610.0,NaN,0.0,NaN,0.0,NaN,0.0,13610.0,no_exact_forest_split_verified,A03544EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,9.658275e+03,324580.0,13610.0,310970.0,9.658275e+03,1.789353e+04,6.560961e+04,2254.358999,1.106930e+04,5.595399e+04,3.653057e+07,FRA mangrove hectares are removed from GCW onl...
4,ASM,American Samoa,ASM,20.0,NaN,0.0,NaN,0.0,NaN,0.0,20.0,no_exact_forest_split_verified,A03274EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,3.346779e+01,15850.0,20.0,15850.0,3.346779e+01,6.200453e+01,2.273499e+02,7.811788,3.835726e+01,1.938914e+02,1.265855e+05,FRA mangrove hectares are removed from GCW onl...
5,NaN,Europa Island,ATF-4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.644516e+02,NaN,NaN,NaN,6.644516e+02,1.231005e+03,4.513685e+03,155.091086,7.615247e+02,3.849416e+03,2.513160e+06,FRA mangrove hectares are removed from GCW onl...
6,NaN,ATF_GI,ATF_GI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,FRA mangrove hectares are removed from GCW onl...
7,ATG,Antigua and Barbuda,ATG,840.0,NaN,0.0,NaN,0.0,NaN,0.0,840.0,no_exact_forest_split_verified,A03276EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,9.526774e+02,7790.0,840.0,7790.0,9.526774e+02,1.764990e+03,6.471631e+03,222.366493,1.091859e+03,5.519215e+03,3.603319e+06,FRA mangrove hectares are removed from GCW onl...
8,AUS,Australia,AUS,1110000.0,NaN,0.0,NaN,0.0,NaN,0.0,1110000.0,no_exact_forest_split_verified,A03280EN,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,1.069096e+06,133561590.0,1110000.0,131504450.0,1.069096e+06,1.980674e+06,7.262472e+06,249539.935500,1.225285e+06,6.193670e+06,4.043649e+09,FRA mangrove hectares are removed from GCW onl...
9,BEN,Benin,BEN,13830.0,NaN,0.0,NaN,0.0,NaN,0.0,13830.0,no_exact_forest_split_verified,A03637FR,https://openknowledge.fao.org/handle/20.500.14...,No matching exact national Forest/OWL area spl...,1.930593e+03,3133270